# Parquet by Experiment: File Size, Speed, and AI Data

This notebook uses real data files to show why Parquet is useful.

You will compare CSV and Parquet using NYC taxi trip data, then inspect Parquet files used for speech and computer vision datasets.

## Learning path

1. Compare CSV and Parquet file size.
2. Run the same analytics questions and compare speed.
3. Inspect Parquet schema, row groups, column chunks, compression, and metadata.
4. See projection pushdown: read only needed columns.
5. See predicate pushdown: skip row groups that cannot match a filter.
6. See partitioning: skip folders and files by date.
7. Inspect speech and computer vision Parquet files.

The main goal is not to memorize internal terms. The goal is to connect each concept to something observable: file size, query time, selected columns, skipped data, or stored metadata.


## 0. Preparation

The CSV and Parquet files contain the same January 2024 yellow taxi trips. They are stored in the `NY` folder next to this notebook.

Required packages:

```text
pip install pandas duckdb pyarrow
```

Parquet is common in analytics systems, data lakes, feature stores, ML training datasets, speech datasets, and computer vision datasets. The reason is the same across these use cases: many workflows need to read selected columns, inspect metadata, filter records, and store large data compactly.


In [89]:
from pathlib import Path
from time import perf_counter
import os

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA_DIR = Path("NY")
CSV_PATH = DATA_DIR / "yellow_tripdata_2024-01.csv"
PARQUET_PATH = DATA_DIR / "yellow_tripdata_2024-01.parquet"
YELLOW_2026_PARQUET_PATH = DATA_DIR / "yellow_tripdata_2026-01.parquet"
FHVHV_2026_CSV_PATH = DATA_DIR / "fhvhv_tripdata_2026-01.csv"
FHVHV_2026_PARQUET_PATH = DATA_DIR / "fhvhv_tripdata_2026-01.parquet"
PARTITIONED_PATH = Path("yellow_taxi_partitioned")
DEMO_OUTPUT_DIR = Path("demo_outputs")
DEMO_OUTPUT_DIR.mkdir(exist_ok=True)

assert CSV_PATH.exists(), f"Missing file: {CSV_PATH.resolve()}"
assert PARQUET_PATH.exists(), f"Missing file: {PARQUET_PATH.resolve()}"
assert YELLOW_2026_PARQUET_PATH.exists(), f"Missing file: {YELLOW_2026_PARQUET_PATH.resolve()}"
assert FHVHV_2026_PARQUET_PATH.exists(), f"Missing file: {FHVHV_2026_PARQUET_PATH.resolve()}"

def mb(path):
    return path.stat().st_size / 1024 / 1024

BENCHMARKS = []

def timed(label, fn):
    start = perf_counter()
    result = fn()
    elapsed = perf_counter() - start
    BENCHMARKS.append({"step": label, "seconds": elapsed})
    print(f"{label}: {elapsed:.3f} seconds")
    return result

print("CSV file:", CSV_PATH)
print("Parquet file:", PARQUET_PATH)


CSV file: NY\yellow_tripdata_2024-01.csv
Parquet file: NY\yellow_tripdata_2024-01.parquet


## 1. Start with CSV: Simple but Expensive at Scale

CSV is useful because it is portable and easy to inspect. However, it has several limitations:

- It stores values as text.
- It does not store a strong built-in schema.
- Readers must parse delimiters and convert text into data types.
- Analytics queries often need only a few columns, but a CSV reader still has to work through each row.
- Compression is not built into the file format.

Start with something concrete: compare the file sizes.


In [90]:
size_comparison = pd.DataFrame({
    "format": ["CSV", "Parquet"],
    "size_mb": [mb(CSV_PATH), mb(PARQUET_PATH)],
})
size_comparison["relative_to_parquet"] = size_comparison["size_mb"] / mb(PARQUET_PATH)
size_comparison.round(2)


,format,size_mb,relative_to_parquet
0,CSV,285.44,5.99
1,Parquet,47.65,1.00


The Parquet file stores the same logical taxi table, but it is much smaller. The smaller size comes from columnar layout, encoding, and compression.

This is the first important observation:

```text
CSV is convenient for humans.
Parquet is designed for machines reading large analytical datasets.
```


A CSV file is readable as plain text. That convenience is useful for inspection, but the reader must infer data types when it loads the file.

Read only five rows first. This is not a speed test yet; it is a reminder that CSV readers must parse text and infer types.


In [91]:
csv_preview = timed("Read five CSV rows", lambda: pd.read_csv(CSV_PATH, nrows=5))
print("Inferred data types:")
display(csv_preview.dtypes.to_frame("dtype"))
csv_preview


Read five CSV rows: 0.008 seconds
Inferred data types:


,dtype
VendorID,int64
tpep_pickup_datetime,object
tpep_dropoff_datetime,object
passenger_count,int64
trip_distance,float64
RatecodeID,int64
store_and_fwd_flag,object
PULocationID,int64
DOLocationID,int64
payment_type,int64


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


### Analytics question 1: one column

Suppose a dashboard needs only one answer: how many trips included an airport fee?

This question only needs the `Airport_fee` column. With CSV, the reader still has to parse text rows. With Parquet, the reader can target the required column.

First run the CSV version.


In [92]:
csv_airport_query = f"""
-- SELECT chooses what the query returns.
SELECT
    -- COUNT(*) counts all rows in the CSV file.
    COUNT(*) AS total_trips,

    -- FILTER counts only rows where the airport fee is greater than 0.
    -- This gives one conditional count in the same query.
    COUNT(*) FILTER (WHERE Airport_fee > 0) AS trips_with_airport_fee

-- read_csv_auto tells DuckDB to read the CSV file and infer column types.
FROM read_csv_auto('{CSV_PATH.as_posix()}')
"""

timed("CSV query", lambda: duckdb.sql(csv_airport_query).df())


CSV query: 0.472 seconds


,total_trips,trips_with_airport_fee
0,2964624,232752


### Analytics question 2: vendor revenue

A manager may ask:

1. How many trips did Vendor 1 complete?
2. How much total revenue did Vendor 1 generate?
3. How do the vendors compare?

These questions need only a few columns, such as `VendorID` and `total_amount`. This is where Parquet's columnar layout becomes useful.

First run the CSV versions.


In [93]:
csv_vendor_1_query = f"""
SELECT
    -- Return the vendor ID so we know which group this row describes.
    VendorID,

    -- Count the number of trips for this vendor.
    COUNT(*) AS trip_count,

    -- Add total_amount for all selected trips, then round to 2 decimals.
    ROUND(SUM(total_amount), 2) AS total_revenue

-- Read the CSV file as a table.
FROM read_csv_auto('{CSV_PATH.as_posix()}')

-- Keep only Vendor 1 rows before aggregation.
WHERE VendorID = 1

-- Because VendorID appears in SELECT with aggregates, we group by VendorID.
GROUP BY VendorID
"""

timed("CSV query: Vendor 1 summary", lambda: duckdb.sql(csv_vendor_1_query).df())


CSV query: Vendor 1 summary: 0.433 seconds


,VendorID,trip_count,total_revenue
0,1,729732,18841261.98


In [94]:
csv_vendor_comparison_query = f"""
SELECT
    -- One output row per vendor.
    VendorID,

    -- Number of trips from that vendor.
    COUNT(*) AS trip_count,

    -- Total revenue from that vendor.
    ROUND(SUM(total_amount), 2) AS total_revenue,

    -- Average revenue per trip from that vendor.
    ROUND(AVG(total_amount), 2) AS average_revenue_per_trip

FROM read_csv_auto('{CSV_PATH.as_posix()}')

-- Collapse many trip rows into one summary row per VendorID.
GROUP BY VendorID

-- Put the vendor with the most trips first.
ORDER BY trip_count DESC
"""

timed("CSV query: compare vendors", lambda: duckdb.sql(csv_vendor_comparison_query).df())


CSV query: compare vendors: 0.424 seconds


,VendorID,trip_count,total_revenue,average_revenue_per_trip
0,2,2234632,60602721.27,27.12
1,1,729732,18841261.98,25.82
2,6,260,12401.03,47.70


Now run the same questions against the Parquet file.

The SQL logic is almost the same. The storage format is different.


In [95]:
parquet_airport_query = f"""
SELECT
    -- Count all trips in the Parquet file.
    COUNT(*) AS total_trips,

    -- Count only trips with Airport_fee > 0.
    COUNT(*) FILTER (WHERE Airport_fee > 0) AS trips_with_airport_fee

-- Same logical query as CSV, but now the source is Parquet.
-- DuckDB can use Parquet metadata and columnar reading.
FROM read_parquet('{PARQUET_PATH.as_posix()}')
"""

timed("Parquet query: airport fee", lambda: duckdb.sql(parquet_airport_query).df())


Parquet query: airport fee: 0.018 seconds


,total_trips,trips_with_airport_fee
0,2964624,232752


In [96]:
parquet_vendor_1_query = f"""
SELECT
    VendorID,
    COUNT(*) AS trip_count,
    ROUND(SUM(total_amount), 2) AS total_revenue

-- Read the same taxi table from Parquet.
FROM read_parquet('{PARQUET_PATH.as_posix()}')

-- Filter to one vendor.
WHERE VendorID = 1

-- Produce one summary row for Vendor 1.
GROUP BY VendorID
"""

timed("Parquet query: Vendor 1 summary", lambda: duckdb.sql(parquet_vendor_1_query).df())


Parquet query: Vendor 1 summary: 0.027 seconds


,VendorID,trip_count,total_revenue
0,1,729732,18841261.98


In [97]:
parquet_vendor_comparison_query = f"""
SELECT
    VendorID,
    COUNT(*) AS trip_count,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS average_revenue_per_trip

FROM read_parquet('{PARQUET_PATH.as_posix()}')

-- Group trips by vendor, just like the CSV version.
GROUP BY VendorID

-- Sort by the aggregate result.
ORDER BY trip_count DESC
"""

timed("Parquet query: compare vendors", lambda: duckdb.sql(parquet_vendor_comparison_query).df())


Parquet query: compare vendors: 0.028 seconds


,VendorID,trip_count,total_revenue,average_revenue_per_trip
0,2,2234632,60602721.27,27.12
1,1,729732,18841261.98,25.82
2,6,260,12401.03,47.70


In [98]:
pd.DataFrame(BENCHMARKS).assign(seconds=lambda x: x["seconds"].round(3))


,step,seconds
0,Read five CSV rows,0.008
1,CSV query,0.472
2,CSV query: Vendor 1 summary,0.433
3,CSV query: compare vendors,0.424
4,Parquet query: airport fee,0.018
5,Parquet query: Vendor 1 summary,0.027
6,Parquet query: compare vendors,0.028


## 2. Row-Oriented vs Columnar Storage

A logical table can be stored in different ways.

| Storage layout | Simplified physical order | Good fit |
|---|---|---|
| Row-oriented | `trip_1: vendor, distance, fee`, then `trip_2: vendor, distance, fee` | Looking up or updating complete records |
| Columnar | all `vendor` values, then all `distance` values, then all `fee` values | Aggregating or filtering a small number of columns |

CSV is naturally row-oriented. Parquet is columnar.

For example, calculating `AVG(trip_distance)` needs only one column. A columnar reader can avoid loading unrelated columns such as timestamps, location IDs, and payment details.


In [99]:
toy_table = pd.DataFrame({
    "VendorID": [1, 2, 2],
    "trip_distance": [1.2, 3.5, 0.8],
    "Airport_fee": [0.0, 1.75, 0.0],
})

print("Logical table:")
display(toy_table)
print("Simplified row-oriented layout:")
print(toy_table.to_dict(orient="records"))
print("\nSimplified columnar layout:")
print(toy_table.to_dict(orient="list"))


Logical table:


,VendorID,trip_distance,Airport_fee
0,1,1.2,0.00
1,2,3.5,1.75
2,2,0.8,0.00


Simplified row-oriented layout:
[{'VendorID': 1, 'trip_distance': 1.2, 'Airport_fee': 0.0}, {'VendorID': 2, 'trip_distance': 3.5, 'Airport_fee': 1.75}, {'VendorID': 2, 'trip_distance': 0.8, 'Airport_fee': 0.0}]

Simplified columnar layout:
{'VendorID': [1, 2, 2], 'trip_distance': [1.2, 3.5, 0.8], 'Airport_fee': [0.0, 1.75, 0.0]}


## 3. Inspect the Parquet File

A Parquet file is more than a table saved in a different extension. It stores schema and metadata that readers can use before reading all values.

A simplified view:

```text
Parquet file
|-- Row group 0
|   |-- Column chunk: VendorID
|   |-- Column chunk: trip_distance
|   `-- ...
|-- Row group 1
|   `-- ...
`-- File metadata: schema, row-group statistics, encodings, compression codecs
```

A **row group** contains a batch of rows. Inside each row group, values are stored by column.


In [100]:
parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print("Rows:", f"{metadata.num_rows:,}")
print("Columns:", metadata.num_columns)
print("Row groups:", metadata.num_row_groups)
print("\nSchema stored in the Parquet file:")
print(parquet_file.schema_arrow)


Rows: 2,964,624
Columns: 19
Row groups: 3

Schema stored in the Parquet file:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double


### Row-group metadata

A **row group** is a horizontal chunk of rows inside a Parquet file. Each row group stores its own column chunks and metadata.

Readers can sometimes use row-group statistics such as minimum and maximum values to decide whether a row group might match a filter. However, statistics are optional. Some Parquet files do not store min/max statistics for every column.

The next cell inspects the pickup timestamp metadata in the original taxi Parquet file. If `pickup_min` and `pickup_max` are `None`, that means this file did not store those statistics. The row groups still exist; they just do not expose useful min/max values for this column.


In [101]:
pickup_col_index = parquet_file.schema_arrow.names.index("tpep_pickup_datetime")
row_group_rows = []

for row_group_id in range(metadata.num_row_groups):
    row_group = metadata.row_group(row_group_id)
    column = row_group.column(pickup_col_index)
    stats = column.statistics
    row_group_rows.append({
        "row_group": row_group_id,
        "rows": row_group.num_rows,
        "has_statistics": stats is not None,
        "pickup_min": stats.min if stats else None,
        "pickup_max": stats.max if stats else None,
        "compression": column.compression,
        "encodings": ", ".join(column.encodings),
    })

pd.DataFrame(row_group_rows)


,row_group,rows,has_statistics,pickup_min,pickup_max,compression,encodings
0,0,1048576,False,None,None,ZSTD,"PLAIN, RLE, RLE_DICTIONARY"
1,1,1048576,False,None,None,ZSTD,"PLAIN, RLE, RLE_DICTIONARY"
2,2,867472,False,None,None,ZSTD,"PLAIN, RLE, RLE_DICTIONARY"


### A tiny row-group statistics demo

The real taxi file above has row groups, but it does not expose pickup min/max statistics. To make the idea visible, create a tiny artificial Parquet file.

This toy table has 12 rows. We write it with `row_group_size=4`, so Parquet creates three row groups. Because the rows are already sorted by `day`, each row group covers a clear date range.


In [102]:
row_group_stats_demo_path = DEMO_OUTPUT_DIR / "row_group_stats_demo.parquet"

row_group_stats_demo = pd.DataFrame({
    "day": pd.to_datetime([
        "2024-01-01", "2024-01-01", "2024-01-02", "2024-01-02",
        "2024-01-10", "2024-01-10", "2024-01-11", "2024-01-11",
        "2024-01-20", "2024-01-20", "2024-01-21", "2024-01-21",
    ]),
    "trip_count": [100, 120, 90, 110, 130, 125, 140, 135, 80, 85, 95, 105],
})

pq.write_table(
    pa.Table.from_pandas(row_group_stats_demo, preserve_index=False),
    row_group_stats_demo_path,
    row_group_size=4,
    compression="zstd",
    write_statistics=True,
)

stats_demo_file = pq.ParquetFile(row_group_stats_demo_path)
stats_demo_metadata = stats_demo_file.metadata
day_demo_col_index = stats_demo_file.schema_arrow.names.index("day")

stats_demo_rows = []
for row_group_id in range(stats_demo_metadata.num_row_groups):
    row_group = stats_demo_metadata.row_group(row_group_id)
    column = row_group.column(day_demo_col_index)
    stats = column.statistics
    stats_demo_rows.append({
        "row_group": row_group_id,
        "rows": row_group.num_rows,
        "day_min": stats.min,
        "day_max": stats.max,
        "compression": column.compression,
    })

pd.DataFrame(stats_demo_rows)


,row_group,rows,day_min,day_max,compression
0,0,4,2024-01-01,2024-01-02,ZSTD
1,1,4,2024-01-10,2024-01-11,ZSTD
2,2,4,2024-01-20,2024-01-21,ZSTD


Now the row-group metadata says something useful:

| Row group | Date range |
|---|---|
| 0 | January 1 to January 2 |
| 1 | January 10 to January 11 |
| 2 | January 20 to January 21 |

For the filter `day = '2024-01-10'`, row groups 0 and 2 cannot contain matching values. Their metadata allows the reader to skip them.

This is the basic idea behind row-group pruning:

```text
Filter condition
  -> compare with row-group min/max metadata
  -> skip row groups that cannot match
  -> read fewer bytes
```


### Compression and dictionary encoding

The metadata above shows two useful ideas:

- **Compression codec:** the file may use a codec such as ZSTD, Snappy, or Gzip. ZSTD often gives strong compression with good speed; Snappy usually prioritizes very fast decompression; Gzip often compresses well but can use more CPU time.
- **Dictionary encoding:** repeated values can be represented by compact identifiers. This is useful for columns with repeated categories, such as `payment_type`.

Columnar storage groups similar values together, which often makes encoding and compression more effective.


In [103]:
payment_col_index = parquet_file.schema_arrow.names.index("payment_type")
payment_column = metadata.row_group(0).column(payment_col_index)

print("Column:", payment_column.path_in_schema)
print("Compression:", payment_column.compression)
print("Encodings:", payment_column.encodings)


Column: payment_type
Compression: ZSTD
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')


### Reading the encoding output

A typical output for `payment_type` might look like this:

```text
Column: payment_type
Compression: ZSTD
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
```

There are two layers to notice:

| Layer | Example | What it means |
|---|---|---|
| Compression codec | `ZSTD` | A general-purpose compression algorithm applied to encoded Parquet pages. |
| Column encoding | `PLAIN`, `RLE`, `RLE_DICTIONARY` | Parquet-specific ways to represent the column values before compression. |

A useful mental model:

```text
original column values
  -> Parquet encoding such as dictionary or RLE
  -> compression codec such as ZSTD
  -> bytes stored in the .parquet file
```

`RLE` means **run-length encoding**. It compresses repeated runs of values by storing the value and the repeat count.

```text
Original:
1, 1, 1, 1, 1, 2, 2, 2, 3, 3

RLE idea:
1 repeated 5 times
2 repeated 3 times
3 repeated 2 times
```

This works especially well when values repeat many times in a row.

`RLE_DICTIONARY` means Parquet is using dictionary encoding, and the dictionary ids can be stored compactly with RLE-style encoding.

For a low-cardinality column such as `payment_type`, the values repeat from a small set of categories. Instead of storing the full value every time, Parquet can build a dictionary:

```text
0 -> Credit card
1 -> Cash
2 -> No charge
3 -> Dispute
```

Then the column can be represented as small integer ids:

```text
0, 0, 0, 1, 1, 0, 2, 2
```

Those ids are much cheaper to store than repeated strings or repeated category values. If the ids repeat, RLE can make them even smaller:

```text
0 repeated 3 times
1 repeated 2 times
0 repeated 1 time
2 repeated 2 times
```

`PLAIN` means some values or dictionary pages are stored in a straightforward representation. Seeing `PLAIN` together with `RLE_DICTIONARY` is normal: Parquet can use different encodings for different parts of the same column chunk, such as the dictionary page and the data pages.

Why this matters for data engineering:

- Category columns such as `payment_type`, `vendor_id`, status codes, and country codes often compress very well.
- Numeric or timestamp columns can use other encodings and statistics to support faster scans.
- Parquet readers can combine column pruning, row-group statistics, encodings, and compression to avoid reading unnecessary bytes.
- Compression such as `ZSTD` reduces file size, while encodings such as dictionary and RLE make the column easier to compress in the first place.

Short classroom version:

> `payment_type` has only a few repeated categories. Parquet can replace those categories with small dictionary ids, use RLE to store repeated ids compactly, and then apply ZSTD compression to the encoded bytes.


### Can we see the dictionary?

Usually, readers return the **decoded logical values** of the column, not the exact internal Parquet dictionary page. For this file, `payment_type` is already an integer code column, so PyArrow reads it as `int64` values such as `1`, `2`, `3`, and `4`.

We can still make the dictionary idea visible by asking Arrow to dictionary-encode the decoded values again. The dictionary order shown below is Arrow's re-encoding for inspection, not a promise that it is byte-for-byte the same dictionary order stored inside the Parquet file.

Read the next cell line by line:

| Code | Meaning |
|---|---|
| `read_row_group(0, columns=["payment_type"])` | Read only row group 0 and only the `payment_type` column. This is a small slice of the Parquet file. |
| `.column("payment_type")` | Get that one column from the Arrow table. |
| `.combine_chunks()` | Combine column chunks into one Arrow array so it is easier to inspect. |
| `.dictionary_encode()` | Build a dictionary from the repeated values and replace each value with a dictionary id. |
| `encoded_payment.dictionary` | The unique values stored in the dictionary. |
| `encoded_payment.indices` | The integer ids that point into the dictionary. |

The important idea is:

```text
original values = dictionary[ids]
```

For example, if Arrow prints:

```text
Arrow dictionary values: [2, 1, 4, 3, 0]
First 20 original values: [2, 1, 1, 1, ...]
First 20 dictionary ids: [0, 1, 1, 1, ...]
```

Then dictionary id `0` means value `2`, and dictionary id `1` means value `1`:

| Dictionary id | Dictionary value | Taxi payment code meaning |
|---|---|---|
| `0` | `2` | Cash |
| `1` | `1` | Credit card |

So the first few ids:

```text
0, 1, 1, 1
```

mean the original values:

```text
2, 1, 1, 1
```

In plain English: Arrow found the repeated values in the column, stored each unique value once in a dictionary, and represented the column as small ids into that dictionary.


In [104]:
payment_table = parquet_file.read_row_group(0, columns=["payment_type"])
payment_values = payment_table.column("payment_type").combine_chunks()
encoded_payment = payment_values.dictionary_encode()

print("Arrow dictionary values:", encoded_payment.dictionary.to_pylist())
print("First 20 original values:", payment_values[:20].to_pylist())
print("First 20 dictionary ids:", encoded_payment.indices[:20].to_pylist())


Arrow dictionary values: [2, 1, 4, 3]
First 20 original values: [2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 1]
First 20 dictionary ids: [0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1]


The taxi dataset stores `payment_type` as numeric codes. If we want human-readable labels, we need a separate data dictionary from the dataset documentation.


In [105]:
payment_type_labels = {
    0: "Unknown / none",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

sample_codes = payment_values[:20].to_pylist()
sample_labels = [payment_type_labels.get(code, f"Code {code}") for code in sample_codes]
encoded_labels = pa.array(sample_labels).dictionary_encode()

print("First 20 labels:", sample_labels)
print("Label dictionary:", encoded_labels.dictionary.to_pylist())
print("First 20 label dictionary ids:", encoded_labels.indices.to_pylist())


First 20 labels: ['Cash', 'Credit card', 'Credit card', 'Credit card', 'Credit card', 'Credit card', 'Credit card', 'Cash', 'Cash', 'Cash', 'Credit card', 'Credit card', 'Credit card', 'Credit card', 'Cash', 'Cash', 'Credit card', 'Cash', 'Credit card', 'Credit card']
Label dictionary: ['Cash', 'Credit card']
First 20 label dictionary ids: [0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1]


## 4. Working with Parquet in Python and DuckDB

Parquet is supported by many data tools. The syntax is usually simple because the schema is stored inside the file.

### Read a Parquet file with DuckDB

DuckDB can query a Parquet file directly. It is not necessary to load the entire file into a Pandas DataFrame first.

```sql
SELECT *
FROM read_parquet('data.parquet')
LIMIT 5;
```

The next example selects only a few columns and rows.


In [106]:
duckdb_parquet_query = f"""
SELECT
    -- Select only four columns, not the full taxi table.
    VendorID,
    tpep_pickup_datetime,
    trip_distance,
    total_amount

-- Query the Parquet file directly. No Pandas DataFrame is required first.
FROM read_parquet('{PARQUET_PATH.as_posix()}')

-- Keep only trips longer than 5 miles.
WHERE trip_distance > 5

-- Return only five rows so the result is easy to inspect.
LIMIT 5
"""

duckdb.sql(duckdb_parquet_query).df()


,VendorID,tpep_pickup_datetime,trip_distance,total_amount
0,2,2024-01-01 00:49:44,10.82,64.95
1,2,2024-01-01 00:26:01,5.44,36.00
2,1,2024-01-01 00:35:16,8.20,85.09
3,1,2024-01-01 00:42:05,23.90,127.94
4,2,2024-01-01 00:20:11,5.88,36.40


### Save a Pandas DataFrame as Parquet

Pandas can convert a DataFrame to Parquet with `to_parquet()`. It can read the file back with `read_parquet()`.

```python
df.to_parquet('data.parquet', index=False, compression='snappy')
df = pd.read_parquet('data.parquet')
```

The following example uses only five rows, so it runs quickly.


In [107]:
sample_df = csv_preview.copy()
pandas_sample_parquet = DEMO_OUTPUT_DIR / "pandas_sample.parquet"
pandas_sample_csv = DEMO_OUTPUT_DIR / "pandas_sample.csv"

# DataFrame -> Parquet
sample_df.to_parquet(pandas_sample_parquet, index=False, compression="snappy")

# Parquet -> DataFrame -> CSV
pandas_round_trip = pd.read_parquet(pandas_sample_parquet)
pandas_round_trip.to_csv(pandas_sample_csv, index=False)

print("Created:", pandas_sample_parquet)
print("Created:", pandas_sample_csv)
pandas_round_trip


Created: demo_outputs\pandas_sample.parquet
Created: demo_outputs\pandas_sample.csv


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


### Convert between CSV and Parquet with DuckDB

DuckDB can also convert files without loading the full dataset into a Pandas DataFrame.

```sql
-- CSV -> Parquet
COPY (SELECT * FROM read_csv_auto('data.csv'))
TO 'data.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY);

-- Parquet -> CSV
COPY (SELECT * FROM read_parquet('data.parquet'))
TO 'data.csv' (FORMAT CSV, HEADER TRUE);
```

For large files, this approach is useful because DuckDB processes the data in batches. The next cell runs the same conversion on a small sample.


In [108]:
duckdb_sample_parquet = DEMO_OUTPUT_DIR / "duckdb_sample.parquet"
duckdb_sample_csv = DEMO_OUTPUT_DIR / "duckdb_sample.csv"

# Remove previous demo outputs so this cell can be run repeatedly.
for path in [duckdb_sample_parquet, duckdb_sample_csv]:
    path.unlink(missing_ok=True)

csv_to_parquet_sql = f"""
-- COPY writes the result of a query into a new file.
COPY (
    -- This inner SELECT reads all columns from the sample CSV file.
    SELECT *
    FROM read_csv_auto('{pandas_sample_csv.as_posix()}')
)
-- TO chooses the output file path.
TO '{duckdb_sample_parquet.as_posix()}'
-- FORMAT PARQUET means the output file will be Parquet.
-- COMPRESSION SNAPPY chooses a fast compression codec.
(FORMAT PARQUET, COMPRESSION SNAPPY)
"""

duckdb.sql(csv_to_parquet_sql)

parquet_to_csv_sql = f"""
-- Convert the sample Parquet file back to CSV.
COPY (
    -- Read all rows and columns from the Parquet file.
    SELECT *
    FROM read_parquet('{duckdb_sample_parquet.as_posix()}')
)
TO '{duckdb_sample_csv.as_posix()}'
-- FORMAT CSV writes a CSV file.
-- HEADER TRUE writes column names in the first row.
(FORMAT CSV, HEADER TRUE)
"""

duckdb.sql(parquet_to_csv_sql)

print("Created:", duckdb_sample_parquet)
print("Created:", duckdb_sample_csv)

verify_sample_sql = f"""
-- Read the new Parquet file to verify that conversion worked.
SELECT *
FROM read_parquet('{duckdb_sample_parquet.as_posix()}')
"""

duckdb.sql(verify_sample_sql).df()


Created: demo_outputs\duckdb_sample.parquet
Created: demo_outputs\duckdb_sample.csv


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


### Other commonly used libraries

| Library | Typical use | Example |
|---|---|---|
| PyArrow | Low-level Parquet access and metadata inspection | `pq.read_table('data.parquet')` |
| Polars | Fast DataFrame operations and lazy queries | `pl.scan_parquet('data.parquet')` |
| Apache Spark | Distributed processing for very large datasets | `spark.read.parquet('data.parquet')` |
| Hugging Face Datasets | ML, speech, and computer vision datasets | Dataset files are often stored as Parquet |

For this notebook, DuckDB, Pandas, and PyArrow are enough. The important point is that Parquet is widely supported across analytics and AI tooling.


## 5. Projection Pushdown

**Projection pushdown** means reading only the columns required by a query. It is also often called **column projection**.

This is one of the most important reasons Parquet is fast for analytics. A dashboard often needs a few metrics, not every column in the source table.

This example needs only three columns:

```text
VendorID, Airport_fee, trip_distance
```

Run the next cell more than once. Timings can vary because of disk caching, but the difference should remain visible.


In [109]:
needed_columns = ["VendorID", "Airport_fee", "trip_distance"]

csv_selected = timed(
    "CSV: read three selected columns",
    lambda: pd.read_csv(CSV_PATH, usecols=needed_columns),
)
parquet_selected = timed(
    "Parquet: read three selected columns",
    lambda: pd.read_parquet(PARQUET_PATH, columns=needed_columns),
)

print("Rows read:", f"{len(parquet_selected):,}")
parquet_selected.head()


CSV: read three selected columns: 1.421 seconds
Parquet: read three selected columns: 0.073 seconds
Rows read: 2,964,624


,VendorID,Airport_fee,trip_distance
0,2,0.0,1.72
1,1,0.0,1.80
2,1,0.0,4.70
3,1,0.0,1.40
4,1,0.0,0.80


## 6. Predicate Pushdown

**Predicate pushdown** means applying a filter as early as possible. For Parquet, a reader can inspect row-group statistics and skip a row group when its minimum and maximum values prove that it cannot match the filter.

The real taxi file contains a few timestamp outliers. These outliers widen the minimum and maximum ranges, so the metadata is less effective for a simple demonstration.

The following small example creates a sorted Parquet file with three row groups.


In [110]:
pushdown_demo_path = DEMO_OUTPUT_DIR / "predicate_pushdown_demo.parquet"
pushdown_demo = pd.DataFrame({
    "day": pd.to_datetime([
        "2024-01-01", "2024-01-01", "2024-01-02", "2024-01-02",
        "2024-01-10", "2024-01-10", "2024-01-11", "2024-01-11",
        "2024-01-20", "2024-01-20", "2024-01-21", "2024-01-21",
    ]),
    "trip_count": [100, 120, 90, 110, 130, 125, 140, 135, 80, 85, 95, 105],
})

pq.write_table(pa.Table.from_pandas(pushdown_demo), pushdown_demo_path, row_group_size=4)
pushdown_file = pq.ParquetFile(pushdown_demo_path)
day_col_index = pushdown_file.schema_arrow.names.index("day")

demo_stats = []
for row_group_id in range(pushdown_file.metadata.num_row_groups):
    row_group = pushdown_file.metadata.row_group(row_group_id)
    stats = row_group.column(day_col_index).statistics
    demo_stats.append({
        "row_group": row_group_id,
        "rows": row_group.num_rows,
        "day_min": stats.min,
        "day_max": stats.max,
    })

pd.DataFrame(demo_stats)


,row_group,rows,day_min,day_max
0,0,4,2024-01-01,2024-01-02
1,1,4,2024-01-10,2024-01-11
2,2,4,2024-01-20,2024-01-21


For the filter `day = '2024-01-10'`, row groups 0 and 2 cannot contain matching values. Their metadata allows the reader to skip them.


In [111]:
filtered_demo = pq.read_table(
    pushdown_demo_path,
    filters=[("day", "=", pd.Timestamp("2024-01-10"))],
).to_pandas()

filtered_demo


,day,trip_count
0,2024-01-10,130
1,2024-01-10,125


## 7. Partitioning by Time

Row groups exist **inside** Parquet files. Partitions are a separate, dataset-level strategy: they organize multiple files into folders.

Time-based partitions are common for logs, transactions, sensor readings, and event data:

```text
yellow_taxi_partitioned/
`-- year=2024/
    `-- month=1/
        |-- day=1/
        |-- day=2/
        `-- ...
```

When a query requests one day, the engine can avoid opening files for other days. This is called **partition pruning**.

> Partition by columns that appear frequently in filters. Avoid creating extremely small partitions.


### Optional: build the partitioned dataset

The prepared folder already contains a partitioned dataset. Set `BUILD_PARTITIONED_DATASET = True` only when you want to rebuild it.


In [112]:
BUILD_PARTITIONED_DATASET = False

if BUILD_PARTITIONED_DATASET:
    if PARTITIONED_PATH.exists():
        raise FileExistsError(
            f"{PARTITIONED_PATH} already exists. Remove or rename it before rebuilding."
        )

    build_partition_query = f"""
    -- COPY writes the query result into a dataset folder.
    COPY (
        SELECT
            -- Keep all original taxi columns.
            *,

            -- Create partition columns from the pickup timestamp.
            YEAR(CAST(tpep_pickup_datetime AS TIMESTAMP)) AS year,
            MONTH(CAST(tpep_pickup_datetime AS TIMESTAMP)) AS month,
            DAY(CAST(tpep_pickup_datetime AS TIMESTAMP)) AS day

        -- Read the original single Parquet file.
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
    )
    -- Write a folder of Parquet files, not one single file.
    TO '{PARTITIONED_PATH.as_posix()}' (
        FORMAT PARQUET,

        -- Create folder levels such as year=2024/month=1/day=1/.
        PARTITION_BY (year, month, day),

        -- Compress each Parquet file with Snappy.
        COMPRESSION SNAPPY
    )
    """
    timed("Build partitioned dataset", lambda: duckdb.sql(build_partition_query))
else:
    print("Using the prepared partitioned dataset:", PARTITIONED_PATH)


Using the prepared partitioned dataset: yellow_taxi_partitioned


In [113]:
partition_files = sorted(PARTITIONED_PATH.rglob("*.parquet"))
assert partition_files, f"No Parquet files found under {PARTITIONED_PATH.resolve()}"

print("Number of partition files:", len(partition_files))
print("Example partitions:")
for path in partition_files[:8]:
    print(" -", path.relative_to(PARTITIONED_PATH))


Number of partition files: 35
Example partitions:
 - year=2002\month=12\day=31\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2009\month=1\day=1\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2023\month=12\day=31\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2024\month=1\day=1\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2024\month=1\day=10\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2024\month=1\day=11\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2024\month=1\day=12\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet
 - year=2024\month=1\day=13\ffe2e60fcebc4e81a1bff89115aba2ae-0.parquet


Query one day of the partitioned dataset. DuckDB reads the `year`, `month`, and `day` values from the folder names and prunes unrelated partitions.


In [114]:
partition_glob = (PARTITIONED_PATH / "**" / "*.parquet").as_posix()
partition_query = f"""
SELECT
    -- These partition columns come from folder names such as year=2024/month=1/day=1.
    year,
    month,
    day,

    -- Count trips for the selected day.
    COUNT(*) AS trip_count,

    -- Average trip distance for the selected day.
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance

-- Read all Parquet files under the partitioned folder.
-- hive_partitioning = true tells DuckDB to parse folder names as columns.
FROM read_parquet('{partition_glob}', hive_partitioning = true)

-- This filter lets DuckDB skip unrelated folders and files.
WHERE year = 2024 AND month = 1 AND day = 1

-- The output has one summary row for this date.
GROUP BY year, month, day
"""

timed("Partitioned query for one day", lambda: duckdb.sql(partition_query).df())


Partitioned query for one day: 0.040 seconds


,year,month,day,trip_count,avg_trip_distance
0,2024,1,1,81013,4.4


## 8. Optional Larger File Check

The `NY` folder also contains larger 2026 files.

This is useful for seeing why Parquet matters more as data grows. The FHVHV CSV file is several GB, so avoid reading the full CSV during a short lesson. Instead, compare file sizes and run a small Parquet-only query.


In [115]:
larger_files = pd.DataFrame([
    {
        "dataset": "Yellow taxi 2024-01",
        "format": "CSV",
        "path": str(CSV_PATH),
        "size_mb": mb(CSV_PATH),
    },
    {
        "dataset": "Yellow taxi 2024-01",
        "format": "Parquet",
        "path": str(PARQUET_PATH),
        "size_mb": mb(PARQUET_PATH),
    },
    {
        "dataset": "Yellow taxi 2026-01",
        "format": "Parquet",
        "path": str(YELLOW_2026_PARQUET_PATH),
        "size_mb": mb(YELLOW_2026_PARQUET_PATH),
    },
    {
        "dataset": "FHVHV 2026-01",
        "format": "CSV",
        "path": str(FHVHV_2026_CSV_PATH),
        "size_mb": mb(FHVHV_2026_CSV_PATH),
    },
    {
        "dataset": "FHVHV 2026-01",
        "format": "Parquet",
        "path": str(FHVHV_2026_PARQUET_PATH),
        "size_mb": mb(FHVHV_2026_PARQUET_PATH),
    },
])

larger_files.assign(size_mb=lambda x: x["size_mb"].round(2))


,dataset,format,path,size_mb
0,Yellow taxi 2024-01,CSV,NY\yellow_tripdata_2024-01.csv,285.44
1,Yellow taxi 2024-01,Parquet,NY\yellow_tripdata_2024-01.parquet,47.65
2,Yellow taxi 2026-01,Parquet,NY\yellow_tripdata_2026-01.parquet,61.19
3,FHVHV 2026-01,CSV,NY\fhvhv_tripdata_2026-01.csv,3355.77
4,FHVHV 2026-01,Parquet,NY\fhvhv_tripdata_2026-01.parquet,482.43


In [116]:
fhvhv_file = pq.ParquetFile(FHVHV_2026_PARQUET_PATH)
fhvhv_metadata = fhvhv_file.metadata

print("Rows:", f"{fhvhv_metadata.num_rows:,}")
print("Columns:", fhvhv_metadata.num_columns)
print("Row groups:", fhvhv_metadata.num_row_groups)
print("Schema:")
print(fhvhv_file.schema_arrow)


Rows: 20,940,373
Columns: 25
Row groups: 20
Schema:
hvfhs_license_num: large_string
dispatching_base_num: large_string
originating_base_num: large_string
request_datetime: timestamp[us]
on_scene_datetime: timestamp[us]
pickup_datetime: timestamp[us]
dropoff_datetime: timestamp[us]
PULocationID: int32
DOLocationID: int32
trip_miles: double
trip_time: int64
base_passenger_fare: double
tolls: double
bcf: double
sales_tax: double
congestion_surcharge: double
airport_fee: double
tips: double
driver_pay: double
shared_request_flag: large_string
shared_match_flag: large_string
access_a_ride_flag: large_string
wav_request_flag: large_string
wav_match_flag: large_string
cbd_congestion_fee: double


In [117]:
fhvhv_selected_query = f"""
SELECT
    -- Count trips in the larger FHVHV Parquet file.
    COUNT(*) AS trip_count,

    -- Average trip distance. This query needs only trip_miles.
    ROUND(AVG(trip_miles), 2) AS avg_trip_miles,

    -- Average passenger fare. This query needs only base_passenger_fare.
    ROUND(AVG(base_passenger_fare), 2) AS avg_base_passenger_fare

-- Read the large Parquet file directly.
-- The query selects only a few columns from a much wider file.
FROM read_parquet('{FHVHV_2026_PARQUET_PATH.as_posix()}')

-- Filter out unusual or invalid negative distances.
WHERE trip_miles >= 0
"""

timed("Large FHVHV Parquet query", lambda: duckdb.sql(fhvhv_selected_query).df())


Large FHVHV Parquet query: 0.060 seconds


,trip_count,avg_trip_miles,avg_base_passenger_fare
0,20940373,4.64,25.65


This optional check is intentionally Parquet-only. Reading the multi-GB CSV can take much longer and distract from the main concept.

The takeaway is simple:

```text
When data grows, reading fewer columns and using metadata matters more.
```


## 9. Speed Checkpoint

Review the timings collected so far. Your exact numbers may differ from another computer because of CPU, disk speed, memory, and file caching.

The pattern matters more than one exact number.


In [118]:
benchmark_df = pd.DataFrame(BENCHMARKS)
benchmark_df["seconds"] = benchmark_df["seconds"].round(3)
benchmark_df


,step,seconds
0,Read five CSV rows,0.008
1,CSV query,0.472
2,CSV query: Vendor 1 summary,0.433
3,CSV query: compare vendors,0.424
4,Parquet query: airport fee,0.018
5,Parquet query: Vendor 1 summary,0.027
6,Parquet query: compare vendors,0.028
7,CSV: read three selected columns,1.421
8,Parquet: read three selected columns,0.073
9,Partitioned query for one day,0.040


Questions to answer from your timings:

1. Which operations were fastest?
2. Which operations were slowest?
3. Did reading selected columns from Parquet avoid unnecessary work?
4. Why can the first run and second run have different timings?

File formats do not work alone. Query engine, disk cache, CPU, compression codec, selected columns, and filters all affect speed.


## 10. Summary

| Topic | Main idea | Notebook example |
|---|---|---|
| CSV limitations | Text parsing, inferred types, larger file size | File-size comparison and CSV query |
| Row vs column storage | Analytics often reads only a few columns | Toy table and selected-column benchmark |
| Row groups and metadata | Parquet stores schema and column statistics | PyArrow metadata inspection |
| Compression and dictionaries | Similar and repeated values can be encoded efficiently | Inspect Snappy and dictionary encoding metadata |
| Projection pushdown | Read only required columns | Read three columns from CSV and Parquet |
| Predicate pushdown | Skip row groups that cannot match a filter | Small sorted Parquet example |
| Time partitioning | Skip unrelated files using folder names | Query one day from a partitioned dataset |

### Final takeaway

> CSV is easy for humans to inspect. Parquet is designed for efficient analytics at scale.


## 11. Parquet for Speech and Computer Vision Data

Parquet is not only for business tables. Modern AI datasets often use Parquet too.

Speech and computer vision datasets may store:

| Dataset type | Typical columns |
|---|---|
| Speech | audio bytes, transcript text, speaker, language, duration |
| Computer vision | image bytes, label text, bounding boxes, OCR text, metadata |

This is useful because a training or inspection script can read labels and metadata without immediately unpacking every audio or image file.

In this folder, two example Parquet files are available:

| Option | File | Binary field | Label field |
|---|---|---|---|
| Speech | `speech/0000.parquet` | `audio.bytes` | `text` |
| Handwriting OCR | `vision/viet-handwriting-ocr-v2/test-00000-of-00001.parquet` | `image.bytes` | `text` |

The vision example is a Vietnamese handwriting OCR dataset. The language is not the point here; it is a convenient small example where Parquet stores image bytes and text labels in one file.


In [119]:
AI_FILES = {
    "speech": Path("speech/0000.parquet"),
    "vision": Path("vision/viet-handwriting-ocr-v2/test-00000-of-00001.parquet"),
}

ai_file_table = pd.DataFrame([
    {"dataset": name, "path": str(path), "exists": path.exists(), "size_mb": mb(path) if path.exists() else None}
    for name, path in AI_FILES.items()
])
ai_file_table


,dataset,path,exists,size_mb
0,speech,speech\0000.parquet,True,447.873387
1,vision,vision\viet-handwriting-ocr-v2\test-00000-of-0...,True,19.677718


### Inspect AI Parquet Metadata

Metadata inspection is safe even when the file contains large audio or image bytes. You can check row counts, schema, row groups, and compression before reading records.

Choose one file by changing `AI_CHOICE`.


In [120]:
AI_CHOICE = "vision"  # choose "speech" or "vision"
AI_PATH = AI_FILES[AI_CHOICE]

ai_parquet = pq.ParquetFile(AI_PATH)
ai_metadata = ai_parquet.metadata

print("File:", AI_PATH)
print("Rows:", f"{ai_metadata.num_rows:,}")
print("Columns:", ai_metadata.num_columns)
print("Row groups:", ai_metadata.num_row_groups)
print("Schema:")
print(ai_parquet.schema_arrow)


File: vision\viet-handwriting-ocr-v2\test-00000-of-00001.parquet
Rows: 1,000
Columns: 3
Row groups: 1
Schema:
image: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
text: string
-- schema metadata --
huggingface: '{"info": {"features": {"image": {"_type": "Image"}, "text":' + 41


### Read Only Labels First

For AI data, labels and metadata are often much smaller than the raw media bytes.

Read only the `text` column first. This is projection pushdown again.


In [121]:
labels_only = timed(
    f"{AI_CHOICE}: read text labels only",
    lambda: pq.read_table(AI_PATH, columns=["text"]).to_pandas(),
)

labels_only.head()


vision: read text labels only: 0.006 seconds


,text
0,trầm tư bên phím đàn. Ông vừa trở về từ lễ tra...
1,"Xe cơ giới phải khắc phục, sửa chữa các khiếm ..."
2,"tiếng đồng hồ sau, phải báo cáo đơn vị"
3,trục Oz và cách trục Oz một khoảng bằng 3. Khi...
4,Hiện nay số lượng cá thể loài sóc bay đen trắng ở


### Read a Few Full Records

Now read a few complete records, including binary audio or image bytes.

Do not unpack the entire dataset unless you really need to. Large media columns can be expensive.


In [122]:
batch = next(ai_parquet.iter_batches(batch_size=3))
records = batch.to_pylist()

for i, record in enumerate(records):
    print("Record", i)
    print("Text:", record.get("text"))
    if AI_CHOICE == "speech":
        audio = record.get("audio")
        print("Audio keys:", list(audio.keys()) if isinstance(audio, dict) else type(audio))
        print("Audio bytes:", len(audio.get("bytes") or b"") if isinstance(audio, dict) else None)
    else:
        image = record.get("image")
        print("Image keys:", list(image.keys()) if isinstance(image, dict) else type(image))
        print("Image bytes:", len(image.get("bytes") or b"") if isinstance(image, dict) else None)
    print()


Record 0
Text: trầm tư bên phím đàn. Ông vừa trở về từ lễ trao giải
Image keys: ['bytes', 'path']
Image bytes: 12050

Record 1
Text: Xe cơ giới phải khắc phục, sửa chữa các khiếm khuyết hư hỏng để kiểm
Image keys: ['bytes', 'path']
Image bytes: 8451

Record 2
Text: tiếng đồng hồ sau, phải báo cáo đơn vị
Image keys: ['bytes', 'path']
Image bytes: 2997



### Optional: Extract a Few Media Files

The records contain bytes. You can write those bytes back to files for inspection.

This does not change the Parquet file. It only extracts a few examples.


In [123]:
EXTRACT_MEDIA = False
extract_dir = DEMO_OUTPUT_DIR / f"{AI_CHOICE}_samples"
extract_dir.mkdir(exist_ok=True)

if EXTRACT_MEDIA:
    for i, record in enumerate(records):
        if AI_CHOICE == "speech":
            media = record["audio"]["bytes"]
            suffix = ".wav"
        else:
            media = record["image"]["bytes"]
            suffix = ".jpg"
        out_path = extract_dir / f"sample_{i}{suffix}"
        out_path.write_bytes(media)
        print("Wrote", out_path, "label:", record.get("text"))
else:
    print("Set EXTRACT_MEDIA = True to write sample files into", extract_dir)


Set EXTRACT_MEDIA = True to write sample files into demo_outputs\vision_samples


### Why AI Datasets Use Parquet

| Need | Why Parquet helps |
|---|---|
| Store many records | Compact columnar storage with compression |
| Keep labels with media | Text labels, metadata, and bytes can live in one table |
| Inspect dataset quickly | Read schema and metadata without unpacking everything |
| Train with selected fields | Read only labels, features, or media columns needed by a step |
| Work with distributed tools | Spark, DuckDB, PyArrow, Polars, and ML dataset tools understand Parquet |

There are two compression layers to remember:

| Layer | Speech example | Vision example | Purpose |
|---|---|---|---|
| Media encoding | WAV, FLAC, MP3 | JPEG, PNG | Store audio or images efficiently |
| Parquet compression | Snappy, Gzip, Zstd | Snappy, Gzip, Zstd | Compress Parquet column chunks |

A JPEG image can already be compressed, but Parquet still helps organize many images and labels into a queryable dataset.


## 12. Parquet Interview Questions

Use the answer structure:

```text
Definition -> key idea -> example -> common trade-off or failure mode
```

### 1. What is the difference between row-oriented and columnar storage?

**Answer:** Row-oriented storage keeps complete rows together and is good for transactional access to full records. Columnar storage keeps values from the same column together and is good for analytical queries that scan only selected columns. CSV is naturally row-oriented, while Parquet is columnar.

### 2. Why is Parquet suitable for analytical data storage?

**Answer:** Parquet is columnar, compressed, schema-aware, and stores metadata such as row-group statistics. This allows query engines to read selected columns, skip irrelevant row groups, and reduce disk I/O for analytical workloads.

### 3. Why can Parquet be much smaller than CSV?

**Answer:** CSV stores values as plain text. Parquet stores typed column values, applies encoding such as dictionary encoding, and compresses column chunks. Similar values are stored together, which usually improves compression.

### 4. What is projection pushdown? Is it the same as column projection?

**Answer:** In this lesson, they refer to the same practical idea: read only the columns needed by a query. `Column projection` describes the logical operation: choose these columns. `Projection pushdown` emphasizes the optimization: push that column selection down into the file reader so unnecessary columns are not read from Parquet at all. For example, a query that needs `VendorID` and `total_amount` does not need to read timestamp, location, or payment detail columns from a Parquet file.

### 5. What is predicate pushdown?

**Answer:** Predicate pushdown means applying filter conditions as early as possible in the storage reading layer. For Parquet, the reader can use metadata such as min/max statistics to skip row groups or files that cannot match the filter.

### 6. What is a Parquet row group?

**Answer:** A row group is a horizontal block of rows inside a Parquet file. Inside each row group, data is stored by column chunks. Row groups allow parallel reading and metadata-based skipping.

### 7. What information is stored in Parquet metadata?

**Answer:** Parquet metadata can include schema, number of rows, row groups, column chunks, compression codec, encodings, offsets, and column statistics such as min and max values. Readers use this metadata to plan efficient reads.

### 8. What is partitioning, and how is it different from row groups?

**Answer:** Row groups are inside a Parquet file. Partitioning organizes a dataset into folders and files based on columns such as `year`, `month`, or `day`. Partitioning helps engines skip entire files or folders before opening them.

### 9. When can partitioning become a problem?

**Answer:** Partitioning can become inefficient when it creates too many tiny files or too many folders. This increases metadata overhead and file-opening cost. Partition columns should match common filters and should not be too high-cardinality.

### 10. Why is Parquet useful for speech and computer vision datasets?

**Answer:** Parquet can store labels, metadata, and binary media bytes in one schema-aware dataset. A script can read only text labels or metadata first, then selectively read audio or image bytes when needed.

### 11. What is the difference between media compression and Parquet compression?

**Answer:** Media compression stores one audio or image object efficiently, such as JPEG, PNG, WAV, FLAC, or MP3. Parquet compression compresses column chunks inside a Parquet file, such as with Snappy, Gzip, or Zstd. A dataset may use both layers.

### 12. A dashboard query on Parquet is still slow. What would you check?

**Answer:** Check whether the query selects too many columns, whether filters can be pushed down, whether partitioning matches the filter pattern, whether there are too many tiny files, whether compression is CPU-heavy, and whether the query engine is scanning more data than expected.
